In [11]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/reference.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/AIMO3_Reference_Problems.pdf
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/sample_submission.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/aimo_3_inference_server.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/aimo_3_gateway.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/__init__.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/templates.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/base_gateway.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/relay.py
/kaggle/input/comp

In [ ]:
# ==========================================
# AIMO3 Gold Medal Contender - SC-TIR Ensemble
# Optimized for Kaggle Limits (Memory & Time)
# ==========================================

# ==========================================
# Step 1: Configuration & Imports
# ==========================================
import os
import re
import sys
import time
import random
import ast
import math
import gc
from dataclasses import dataclass
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Optional
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import sympy as sp
from sympy import *
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    set_seed
)
from concurrent.futures import ThreadPoolExecutor
import contextlib
from io import StringIO

# ==========================================
# Configuration Class
# ==========================================
@dataclass
class AIMOConfig:
    # Model settings - Using both for ensembling sequentially to avoid OOM
    models: Tuple[str, ...] = (
        "Qwen/Qwen2.5-Math-7B-Instruct",
        "deepseek-ai/DeepSeek-Math-7B-Instruct"
    )
    
    # Generation settings
    max_length: int = 2048
    max_new_tokens: int = 1024
    # Dynamic temperatures to balance precision and diversity
    temperatures: Tuple[float, ...] = (0.5, 0.7, 0.9)
    top_p: float = 0.95
    
    # Self-consistency settings
    samples_per_prompt_per_temp: int = 2
    max_code_executions: int = 4
    
    # Answer constraints
    answer_mod: int = 100000
    min_answer: int = 0
    max_answer: int = 99999
    
    # Time & safety
    max_time_minutes: int = 510  # 8.5 hours (Buffer for safe saving)
    code_timeout: int = 10     # Reduced to 10s to save overall time
    use_gpu: bool = True

cfg = AIMOConfig()

set_seed(42)
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print("✅ Configuration loaded")

# ==========================================
# Step 2: Competition Data Loading
# ==========================================
def safe_load_test_data() -> pd.DataFrame:
    candidates = [
        "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv",
        "/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv"
    ]
    
    for path in candidates:
        if os.path.exists(path):
            df = pd.read_csv(path)
            print(f"✅ Loaded test data from: {path}")
            return df
    
    # Fallback dummy data for local testing
    print("⚠️  Warning: test.csv not found. Using dummy data for testing.")
    return pd.DataFrame({
        'id': ['test_1', 'test_2'],
        'problem': ['What is 2+2?', 'Find the last two digits of 3^100.']
    })

test_df = safe_load_test_data()
problem_col = next((col for col in ['problem', 'question', 'prompt'] if col in test_df.columns), None)
assert problem_col is not None, "No problem text column found"

# ==========================================
# Step 3: Safe Code Executor (TIR Core)
# ==========================================
class SafeExecutor:
    def __init__(self, timeout: int = 10):
        self.timeout = timeout
        self.safe_globals = {
            '__builtins__': __builtins__,
            'math': math, 'sympy': sp, 'np': np,
            'pi': math.pi, 'e': math.e
        }
        exec("from sympy import *", self.safe_globals)
    
    def execute_code(self, code: str) -> Tuple[str, bool]:
        """Execute Python code safely with timeout limit."""
        stdout = StringIO()
        stderr = StringIO()
        
        def target():
            with contextlib.redirect_stdout(stdout), contextlib.redirect_stderr(stderr):
                try:
                    exec(code, self.safe_globals.copy())
                except Exception as e:
                    print(f"EXEC_ERROR: {type(e).__name__}: {str(e)}", file=stderr)
        
        try:
            with ThreadPoolExecutor(max_workers=1) as executor:
                future = executor.submit(target)
                future.result(timeout=self.timeout)
            output = stdout.getvalue().strip() or stderr.getvalue().strip()
            # Truncate overly long outputs
            return output[:1000] + ("\n...[truncated]" if len(output)>1000 else ""), True
        except Exception as e:
            return f"TIMEOUT_OR_ERROR: {str(e)}", False

executor = SafeExecutor(cfg.code_timeout)

# ==========================================
# Step 4: Prompt Engineering
# ==========================================
# Emphasizing different roles to increase path diversity
PROMPT_TEMPLATES = [
    """<|im_start|>system
You are an expert IMO competitor. Solve the problem step-by-step.
Write Python code in ```python blocks and execute it for calculations.
Always enclose your final integer answer (0-99999) in \\boxed{{}}.<|im_end|>
<|im_start|>user
{problem}<|im_end|>""",
    
    """<|im_start|>system
You are a brilliant computational mathematician. 
Use algebraic manipulation first, then verify using Python and SymPy.
Your final answer must be a positive integer modulo 100000, enclosed in \\boxed{{ans}}.<|im_end|>
<|im_start|>user
{problem}<|im_end|>"""
]

def generate_prompts(problem: str) -> List[str]:
    return [t.format(problem=problem.strip()) for t in PROMPT_TEMPLATES]

# ==========================================
# Step 5: Answer Extraction
# ==========================================
def parse_candidate(candidate: str) -> Optional[int]:
    try:
        # Check pure digits
        if re.match(r'^\d+$', candidate):
            val = int(candidate) % cfg.answer_mod
            return val if cfg.min_answer <= val <= cfg.max_answer else None
        
        # Check symbolic expression
        expr = sp.sympify(candidate.replace('\,', ''))
        if expr.is_real and float(expr).is_integer():
            val = int(float(expr)) % cfg.answer_mod
            return val if cfg.min_answer <= val <= cfg.max_answer else None
    except:
        pass
    return None

def extract_answers(text: str) -> List[int]:
    answers = []
    # Match standard \boxed{...} format
    for match in re.findall(r'\\boxed\{([^}]+)\}', text, re.IGNORECASE):
        ans = parse_candidate(match.strip())
        if ans is not None:
            answers.append(ans)
            
    # Secondary heuristic: Last integer in the text if no box found
    if not answers:
        numbers = re.findall(r'\b(\d{1,5})\b', text[-300:])
        for num in reversed(numbers):
            ans = parse_candidate(num)
            if ans is not None:
                answers.append(ans)
                break # Only take the very last one
    return answers

# ==========================================
# Step 6: Memory-Efficient Inference Engine
# ==========================================
class AIMOReasoner:
    def __init__(self, model_path: str):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"\n[{time.strftime('%H:%M:%S')}] Loading model: {model_path}")
        
        self.tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
            
        # Optimization: Use float16 and SDPA for memory/speed on Kaggle T4s
        self.model = AutoModelForCausalLM.from_pretrained(
            model_path,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True,
            attn_implementation="sdpa" # Use "flash_attention_2" if using Ampere/A100
        )
        self.model.eval()
    
    def generate_response(self, prompt: str, temperature: float) -> str:
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        input_length = inputs['input_ids'].shape[1]
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=cfg.max_new_tokens,
                temperature=temperature,
                top_p=cfg.top_p,
                do_sample=(temperature > 0.0),
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )
        
        response = self.tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
        return response.strip()
    
    def solve(self, problem: str, global_start_time: float) -> List[int]:
        candidates = []
        prompts = generate_prompts(problem)
        
        for prompt in prompts:
            for temp in cfg.temperatures:
                for _ in range(cfg.samples_per_prompt_per_temp):
                    # Time check constraint
                    if (time.time() - global_start_time) / 60 > cfg.max_time_minutes:
                        return candidates
                    
                    response = self.generate_response(prompt, temp)
                    
                    # Tool Integration Loop
                    code_blocks = re.findall(r'```python\s*(.*?)\s*```', response, re.DOTALL)
                    for code in code_blocks[-cfg.max_code_executions:]:
                        out, success = executor.execute_code(code.strip())
                        if success:
                            feedback = f"Code output:\n{out}\n\nContinue your reasoning and provide the final answer in \\boxed{{}}:"
                            response += self.generate_response(prompt + response + feedback, temp)
                            
                    answers = extract_answers(response)
                    candidates.extend(answers)
                    
                    # Memory cleanup inside tight loop
                    torch.cuda.empty_cache()
                    
        return candidates

def clean_memory():
    """Aggressively clear RAM and VRAM"""
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.ipc_collect()

# ==========================================
# Step 7: Main Pipeline (Sequential Loading)
# ==========================================
print("🚀 Starting AIMO3 Gold Medal Ensemble Pipeline...")
global_start = time.time()

# Store all candidates globally across models
global_candidates: Dict[str, List[int]] = defaultdict(list)

for model_path in cfg.models:
    try:
        # Load model
        reasoner = AIMOReasoner(model_path)
        
        # Process all problems with current model
        for idx, row in test_df.iterrows():
            problem_id = row['id']
            problem_text = row[problem_col]
            
            print(f"  → [Model: {model_path.split('/')[-1]}] Solving ID={problem_id}")
            model_answers = reasoner.solve(problem_text, global_start)
            global_candidates[problem_id].extend(model_answers)
            
            # Time limit check
            if (time.time() - global_start) / 60 > cfg.max_time_minutes:
                print("⏰ Time limit reached during inference!")
                break
                
    except Exception as e:
        print(f"🔥 Error with model {model_path}: {e}")
        
    finally:
        # Unload model from VRAM to make space for the next one
        if 'reasoner' in locals():
            del reasoner.model
            del reasoner.tokenizer
            del reasoner
        clean_memory()
        print(f"🧹 Memory cleared after {model_path}")

    # Check time limit before loading next model
    if (time.time() - global_start) / 60 > cfg.max_time_minutes:
        break

# ==========================================
# Step 8: Majority Voting & Submission
# ==========================================
results = []
for problem_id in test_df['id']:
    candidates = global_candidates.get(problem_id, [])
    valid_answers = [a for a in candidates if a is not None]
    
    if valid_answers:
        vote_counts = Counter(valid_answers)
        best_answer, best_count = vote_counts.most_common(1)[0]
        confidence = best_count / len(valid_answers)
    else:
        best_answer, confidence = 0, 0.0 # Fallback
        
    results.append({
        'id': problem_id,
        'answer': int(best_answer),
        'confidence': confidence,
        'total_votes': len(valid_answers)
    })

results_df = pd.DataFrame(results)
submission_df = test_df[['id']].merge(results_df[['id', 'answer']], on='id', how='left')
submission_df['answer'] = submission_df['answer'].fillna(0).astype(int)

output_path = "submission.csv"
submission_df.to_csv(output_path, index=False)

print("\n" + "="*60)
print("🎉 SUBMISSION READY!")
print(submission_df.head())
print("\n" + "="*60)
print(f"Total runtime: {(time.time() - global_start)/60:.1f} minutes")

NameError: name 'dataclass' is not defined